# Working Notebook for Querying Player Metrics and Standardizing Against Conference Baselines


In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [67]:
file_paths = glob.glob('/Users/longer/uconn/player_data/*.csv')
dataframes = [pd.read_csv(file, index_col='Team') for file in file_paths]

player_df = pd.concat(dataframes, axis=1)
player_df = player_df.loc[:, ~player_df.columns.duplicated()]
player_df = player_df.reset_index()
player_df.to_csv("master_player.csv", index=False)

In [68]:
df = pd.read_csv('/Users/longer/uconn/player_data/master_player.csv')

df

,Team,Player,Position,TOI (sec),TOI (min),Total Games played,Total TOI/GP (sec),Total TOI/GP (min),Successful OZ Offensive Touches,Failed OZ Offensive Touches,...,Defensive Dump-In Recovery Exit Rate,Total Dump Out Attempts,Dump Out Rate,Successful Dump Out Attempts,Dump Out Success Rate,Total Icings,Successful OZ Body Checks,Successful DZ Body Checks,Successful NZ Body Checks,Successful Body Checks
0,Stonehill College Skyhawks,Zach Aben,F,8113.700000,135:14,8,1014.212500,16:54,54,45,...,0.545455,15,0.288462,11,0.733333,2,2,2,0,4
1,College of the Holy Cross Crusaders,Michael Abgrall,F,32080.833333,534:41,37,867.049550,14:27,343,189,...,0.576923,60,0.281690,44,0.733333,7,4,3,1,8
2,Ohio State University Buckeyes,Chris Able,D,36097.300000,601:37,33,1093.857576,18:14,133,86,...,0.756410,102,0.432203,70,0.686275,10,0,9,0,9
3,Michigan Tech University Huskies,Ryan Abraham,F,4624.800000,77:05,6,770.800000,12:51,38,36,...,1.000000,5,0.178571,4,0.800000,0,1,0,0,1
4,University of Alaska Anchorage Seawolves,Logan Acheson,D,37018.366667,616:58,33,1121.768687,18:42,160,141,...,0.743243,106,0.368056,69,0.650943,12,0,7,0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1543,University of North Dakota Fighting Hawks,Bennett Zmolek,D,30788.333333,513:08,30,1026.277778,17:06,116,105,...,0.805556,44,0.305556,31,0.704545,7,0,13,2,15
1544,Army Black Knights,Easton Zueger,D,30545.533333,509:06,30,1018.184444,16:58,109,130,...,0.647436,149,0.433140,99,0.664430,21,0,14,0,14
1545,University of New Hampshire Wildcats,Conner de Haro,D,43242.933333,720:43,33,1310.391919,21:50,230,156,...,0.750000,142,0.376658,95,0.669014,18,1,7,2,10
1546,Robert Morris University Colonials,Luke van Why,D,35588.333333,593:08,31,1148.010753,19:08,250,157,...,0.722892,108,0.355263,70,0.648148,14,0,15,2,17


In [69]:
df.dtypes


Team                          object
Player                        object
Position                      object
TOI (sec)                    float64
TOI (min)                     object
                              ...   
Total Icings                   int64
Successful OZ Body Checks      int64
Successful DZ Body Checks      int64
Successful NZ Body Checks      int64
Successful Body Checks         int64
Length: 394, dtype: object

In [70]:
# lets set a TOI qualifier for players who have played more than 200 mins

df = df[df['TOI (sec)'] > 12000]


In [71]:
len(df)

1220

In [72]:
team_df = pd.read_csv('/Users/longer/uconn/team_data/ss_cols_df.csv')

team_df.columns

Index(['Unnamed: 0', 'Team', 'Conference', 'Successful OZ Open Ice Dekes',
       'Failed OZ Open Ice Dekes', 'Successful Open Ice Dekes',
       'Failed Open Ice Dekes', 'Dump In Rate', 'Total Dump In Attempts',
       'Dump Ins Below Hash Marks',
       ...
       'Successful OZ Blocked Passes', 'Successful NZ Blocked Passes',
       'Successful Blocked Passes', 'DZ Reception Preventions',
       'Total Reception Preventions', 'Total OZ Obstruction Penalties',
       'Total OZ Penalties', 'Total DZ Obstruction Penalties',
       'Total NZ Undisciplined Penalties', 'Total Obstruction Penalties'],
      dtype='object', length=179)

In [73]:
id_cols = ['Player', 'Team', 'Position', 'TOI (min)']

desired_cols = id_cols + list(team_df.columns)

cols_to_keep = player_df.columns.intersection(desired_cols)

player_df = player_df[cols_to_keep]

player_df.head()

,Team,Player,Position,TOI (min),Successful OZ Offensive Touches,Failed OZ Offensive Touches,Failed OZ Possessions,Successful DZ Offensive Touches,Failed DZ Offensive Touches,DZ Offensive Touches Success Rate,...,Controlled Exit with Play After Rate,Successful Defensive Dump-In Recoveries,Failed Defensive Dump-In Recoveries,Total Dump Out Attempts,Dump Out Rate,Successful Dump Out Attempts,Total Icings,Successful OZ Body Checks,Successful DZ Body Checks,Successful Body Checks
0,Stonehill College Skyhawks,Zach Aben,F,135:14,54,45,31,45,21,0.681818,...,0.666667,6,5,15,0.288462,11,2,2,2,4
1,College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,343,189,107,209,74,0.738516,...,0.750000,15,11,60,0.281690,44,7,4,3,8
2,Ohio State University Buckeyes,Chris Able,D,601:37,133,86,42,562,187,0.750334,...,0.783784,118,38,102,0.432203,70,10,0,9,9
3,Michigan Tech University Huskies,Ryan Abraham,F,77:05,38,36,20,20,11,0.645161,...,0.684211,2,0,5,0.178571,4,0,1,0,1
4,University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,160,141,77,546,213,0.719368,...,0.766667,110,38,106,0.368056,69,12,0,7,7


In [74]:
player_df.shape

(1548, 172)

In [75]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

id_cols = ['Team', 'Player', 'Position', 'TOI (min)']
player_df.set_index(id_cols, inplace=True)

scaled_data = scaler.fit_transform(player_df)

player_scaled = pd.DataFrame(scaled_data, index=player_df.index, columns = player_df.columns)

player_scaled.head()

,,,,Successful OZ Offensive Touches,Failed OZ Offensive Touches,Failed OZ Possessions,Successful DZ Offensive Touches,Failed DZ Offensive Touches,DZ Offensive Touches Success Rate,Failed DZ Possessions,Successful NZ Offensive Touches,Failed NZ Offensive Touches,Failed NZ Possessions,...,Controlled Exit with Play After Rate,Successful Defensive Dump-In Recoveries,Failed Defensive Dump-In Recoveries,Total Dump Out Attempts,Dump Out Rate,Successful Dump Out Attempts,Total Icings,Successful OZ Body Checks,Successful DZ Body Checks,Successful Body Checks
Team,Player,Position,TOI (min),,,,,,,,,,,,,,,,,,,,,
Stonehill College Skyhawks,Zach Aben,F,135:14,-1.028896,-1.068593,-0.994559,-0.922229,-0.942070,-0.116737,-0.901918,-1.319885,-1.205387,-1.272551,...,-0.978175,-0.651678,-0.484882,-1.035419,0.155515,-1.018740,-0.725555,0.053540,-0.564963,-0.643342
College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,0.644482,0.449653,0.416347,-0.045624,-0.256116,0.774085,-0.021295,0.320171,0.304936,0.609887,...,-0.090899,-0.482237,-0.164479,0.065895,0.088553,0.191495,0.062959,0.949538,-0.393749,-0.013428
Ohio State University Buckeyes,Chris Able,D,601:37,-0.571468,-0.636314,-0.790349,1.841215,1.206389,0.959765,1.094161,-0.304612,-0.730714,-0.268584,...,0.268808,1.456926,1.277335,1.093789,1.576964,1.145014,0.536067,-0.842458,0.633537,0.144050
Michigan Tech University Huskies,Ryan Abraham,F,77:05,-1.121540,-1.163483,-1.198769,-1.055857,-1.071495,-0.692682,-1.078043,-1.429223,-1.421148,-1.523543,...,-0.791380,-0.726985,-0.751885,-1.280155,-0.931178,-1.275457,-1.040960,-0.394459,-0.907392,-1.115777
University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,-0.415131,-0.056429,-0.140589,1.755692,1.542894,0.473231,1.563827,-0.023460,-0.040281,0.233400,...,0.086556,1.306311,1.277335,1.191683,0.942612,1.108340,0.851473,-0.842458,0.291109,-0.170907


# Create Conference/Team Dictionary

In [76]:

# define teams in each conference
AHA = [
    'Air Force Academy Falcons', 
    'Army Black Knights',
    'Bentley University Falcons',
    'Canisius College Golden Griffins',
    'College of the Holy Cross Crusaders',
    'Mercyhurst University Lakers',
    'Niagara University Purple Eagles',
    'Robert Morris University Colonials',
    'Rochester Institute of Technology Tigers',
    'Sacred Heart University Pioneers'
]

CCHA = [
    'Augustana University Vikings',
    'Bemidji State University Beavers',
    'Bowling Green State University Falcons',
    'Ferris State University Bulldogs',
    'Lake Superior State University Lakers',
    'Michigan Tech University Huskies',
    'Minnesota State University Mavericks',
    'Northern Michigan University Wildcats',
    'The University of St. Thomas Tommies'
]

ECAC = [
    'Brown University Bears',
    'Clarkson University Golden Knights',
    'Colgate University Raiders',
    'Cornell University Big Red',
    'Dartmouth College Big Green',
    'Harvard University Crimson',
    'Princeton University Tigers',
    'Quinnipiac University Bobcats',
    'Rensselaer Polytechnic Institute Engineers',
    'St. Lawrence University Saints',
    'Union College Garnet Chargers',
    'Yale University Bulldogs'
]

HE = [
    'Boston College Eagles',
    'Boston University Terriers',
    'Merrimack College Warriors',
    'Northeastern University Huskies',
    'UMass Lowell River Hawks',
    'UMass Minutemen',
    'University of Connecticut Huskies',
    'University of Maine Black Bears',
    'University of New Hampshire Wildcats',
    'University of Vermont Catamounts',
    'Providence College Friars'
]

NCHC = [
    'Arizona State University Sun Devils',
    'Colorado College Tigers',
    'University of Denver Pioneers',
    'Miami University (Ohio) RedHawks',
    'University of Minnesota-Duluth Bulldogs',
    'University of North Dakota Fighting Hawks',
    'University of Nebraska Omaha Mavericks',
    'St. Cloud State University Huskies',
    'Western Michigan University Broncos'
]

BT = [
    'Michigan State University Spartans',
    'Michigan Wolverines',
    'Ohio State University Buckeyes',
    'Penn State University Nittany Lions'
    'University Of Notre Dame Fighting Irish',
    'University of Minnesota Golden Gophers',
    'University of Wisconsin Badgers'
]

INDE = [
    'University of Alaska Anchorage Seawolves',
    'University of Alaska Fairbanks Nanooks',
    'Lindenwood University Lions',
    'Long Island University Sharks',
    'Stonehill College Skyhawks'
]

In [77]:
conference_mapping = {}
lists = [AHA, CCHA, ECAC, HE, NCHC, BT, INDE]
names = ['Atlantic Hockey', 'CCHA', 'ECAC', 'Hockey East', 'NCHC', 'Big Ten', 'Independent']
for conf_list, conf_name in zip(lists, names):
    for team in conf_list:
        conference_mapping[team] = conf_name

player_scaled = player_scaled.reset_index()
player_scaled['conference'] = player_scaled['Team'].map(conference_mapping)

player_scaled.head()

,Team,Player,Position,TOI (min),Successful OZ Offensive Touches,Failed OZ Offensive Touches,Failed OZ Possessions,Successful DZ Offensive Touches,Failed DZ Offensive Touches,DZ Offensive Touches Success Rate,...,Successful Defensive Dump-In Recoveries,Failed Defensive Dump-In Recoveries,Total Dump Out Attempts,Dump Out Rate,Successful Dump Out Attempts,Total Icings,Successful OZ Body Checks,Successful DZ Body Checks,Successful Body Checks,conference
0,Stonehill College Skyhawks,Zach Aben,F,135:14,-1.028896,-1.068593,-0.994559,-0.922229,-0.942070,-0.116737,...,-0.651678,-0.484882,-1.035419,0.155515,-1.018740,-0.725555,0.053540,-0.564963,-0.643342,Independent
1,College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,0.644482,0.449653,0.416347,-0.045624,-0.256116,0.774085,...,-0.482237,-0.164479,0.065895,0.088553,0.191495,0.062959,0.949538,-0.393749,-0.013428,Atlantic Hockey
2,Ohio State University Buckeyes,Chris Able,D,601:37,-0.571468,-0.636314,-0.790349,1.841215,1.206389,0.959765,...,1.456926,1.277335,1.093789,1.576964,1.145014,0.536067,-0.842458,0.633537,0.144050,Big Ten
3,Michigan Tech University Huskies,Ryan Abraham,F,77:05,-1.121540,-1.163483,-1.198769,-1.055857,-1.071495,-0.692682,...,-0.726985,-0.751885,-1.280155,-0.931178,-1.275457,-1.040960,-0.394459,-0.907392,-1.115777,CCHA
4,University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,-0.415131,-0.056429,-0.140589,1.755692,1.542894,0.473231,...,1.306311,1.277335,1.191683,0.942612,1.108340,0.851473,-0.842458,0.291109,-0.170907,Independent


In [78]:
conference_col = player_scaled.pop('conference')
player_scaled.insert(1, 'Conference', conference_col)
player_scaled.head()

,Team,Conference,Player,Position,TOI (min),Successful OZ Offensive Touches,Failed OZ Offensive Touches,Failed OZ Possessions,Successful DZ Offensive Touches,Failed DZ Offensive Touches,...,Controlled Exit with Play After Rate,Successful Defensive Dump-In Recoveries,Failed Defensive Dump-In Recoveries,Total Dump Out Attempts,Dump Out Rate,Successful Dump Out Attempts,Total Icings,Successful OZ Body Checks,Successful DZ Body Checks,Successful Body Checks
0,Stonehill College Skyhawks,Independent,Zach Aben,F,135:14,-1.028896,-1.068593,-0.994559,-0.922229,-0.942070,...,-0.978175,-0.651678,-0.484882,-1.035419,0.155515,-1.018740,-0.725555,0.053540,-0.564963,-0.643342
1,College of the Holy Cross Crusaders,Atlantic Hockey,Michael Abgrall,F,534:41,0.644482,0.449653,0.416347,-0.045624,-0.256116,...,-0.090899,-0.482237,-0.164479,0.065895,0.088553,0.191495,0.062959,0.949538,-0.393749,-0.013428
2,Ohio State University Buckeyes,Big Ten,Chris Able,D,601:37,-0.571468,-0.636314,-0.790349,1.841215,1.206389,...,0.268808,1.456926,1.277335,1.093789,1.576964,1.145014,0.536067,-0.842458,0.633537,0.144050
3,Michigan Tech University Huskies,CCHA,Ryan Abraham,F,77:05,-1.121540,-1.163483,-1.198769,-1.055857,-1.071495,...,-0.791380,-0.726985,-0.751885,-1.280155,-0.931178,-1.275457,-1.040960,-0.394459,-0.907392,-1.115777
4,University of Alaska Anchorage Seawolves,Independent,Logan Acheson,D,616:58,-0.415131,-0.056429,-0.140589,1.755692,1.542894,...,0.086556,1.306311,1.277335,1.191683,0.942612,1.108340,0.851473,-0.842458,0.291109,-0.170907


# Data Cleaning which includes alignment, column formatting, and dtype standardizing

In [79]:
player_scaled['Conference'].value_counts()

Conference
ECAC               294
Hockey East        266
Atlantic Hockey    257
CCHA               226
NCHC               217
Independent        128
Big Ten            114
Name: count, dtype: int64

In [80]:
conference_baseline = pd.read_csv('/Users/longer/uconn/team_data/cleaned_conference_baselines.csv')

conference_baseline


,Conference,Unnamed: 0.1,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,Successful Open Ice Dekes,Failed Open Ice Dekes,Dump In Rate,Total Dump In Attempts,Dump Ins Below Hash Marks,Total Chip In Attempts,...,Successful NZ Blocked Passes,Successful Blocked Passes,DZ Reception Preventions,Total Reception Preventions,Total OZ Obstruction Penalties,Total OZ Penalties,Total DZ Obstruction Penalties,Total NZ Undisciplined Penalties,Total Obstruction Penalties,Unnamed: 0
0,Atlantic Hockey,19.800000,0.012793,0.044063,0.216652,0.286144,0.463172,0.561241,0.514828,0.475375,...,-0.021864,0.042433,0.145471,-0.131051,0.449798,0.409561,0.714577,-0.093930,0.837592,-0.615918
1,Big Ten,39.857143,0.962282,0.733003,0.836184,0.619693,-1.301132,-0.487698,-0.471846,-0.752269,...,0.407089,0.576650,0.487216,0.635648,0.259444,0.721831,-0.314219,0.643356,-0.008303,0.487078
2,CCHA,20.000000,-0.193569,-0.239019,-0.130594,-0.116506,0.707995,0.732614,0.682087,0.609535,...,0.478901,0.670884,0.962295,0.916047,-0.823454,-0.599782,0.498416,0.167273,-0.048157,-0.604919
3,ECAC,28.250000,-0.454442,-0.198017,-0.466706,-0.144880,0.001229,-0.489008,-0.505586,-0.399811,...,-0.662549,-0.759222,-0.679242,-0.553965,-0.290465,-0.611987,-0.027765,-0.598321,-0.309696,-0.151230
4,Hockey East,37.454545,0.202873,-0.020888,0.140487,-0.171655,-0.204844,-0.211968,-0.122970,-0.048272,...,-0.111981,-0.107158,-0.085905,-0.232441,0.183303,0.156920,-0.241082,-0.283077,-0.216176,0.354953
5,Independent,36.600000,-0.867945,-1.278178,-0.982803,-1.556967,0.498194,-0.717030,-0.737938,-0.460608,...,-0.632772,-1.373303,-0.826707,-0.857115,-0.669479,0.112982,-1.094349,1.176055,-1.191958,0.307959
6,NCHC,40.222222,0.271071,0.619598,0.236074,0.584541,-0.238689,0.332539,0.347250,0.295342,...,0.600570,0.739676,-0.032943,0.234061,0.657071,0.144715,-0.108351,-0.072914,0.463299,0.507155


In [81]:
player_conference_baselines = player_scaled.groupby('Conference').mean(numeric_only=True)
player_conference_baselines

,Successful OZ Offensive Touches,Failed OZ Offensive Touches,Failed OZ Possessions,Successful DZ Offensive Touches,Failed DZ Offensive Touches,DZ Offensive Touches Success Rate,Failed DZ Possessions,Successful NZ Offensive Touches,Failed NZ Offensive Touches,Failed NZ Possessions,...,Controlled Exit with Play After Rate,Successful Defensive Dump-In Recoveries,Failed Defensive Dump-In Recoveries,Total Dump Out Attempts,Dump Out Rate,Successful Dump Out Attempts,Total Icings,Successful OZ Body Checks,Successful DZ Body Checks,Successful Body Checks
Conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,-0.066320,-0.062870,-0.057518,0.003647,0.019504,-0.011650,0.003376,-0.020664,-0.010561,-0.063493,...,0.001965,0.008726,0.034786,-0.002764,0.017649,-0.003433,-0.002086,0.039594,-0.002021,0.012307
Big Ten,0.211941,0.210393,0.230865,0.101884,0.052915,0.180144,0.076037,0.211108,0.187592,0.251013,...,0.108128,0.038474,-0.023483,-0.009458,-0.227258,-0.013106,0.062959,0.088908,0.252060,0.308435
CCHA,-0.039919,-0.044952,-0.022055,0.013670,0.017452,0.051482,0.019229,-0.010674,-0.005530,0.002954,...,0.085231,0.015425,0.016043,0.053984,0.034468,0.065895,-0.011706,0.132832,0.075954,0.109210
ECAC,-0.025984,-0.042837,-0.061911,-0.043933,-0.001844,-0.160878,-0.017501,-0.052202,-0.006963,0.005458,...,-0.260248,-0.026807,-0.005186,-0.075536,-0.052396,-0.094037,0.059740,-0.040936,-0.072285,-0.106094
Hockey East,-0.003437,0.034617,0.035425,0.041426,-0.006365,0.085349,0.004969,-0.028744,-0.061695,-0.061469,...,0.127090,0.019081,0.022423,0.038846,0.037571,0.040388,-0.015893,0.088908,0.002748,0.055838
Independent,-0.260016,-0.280146,-0.290701,-0.221805,-0.188269,-0.134684,-0.150866,-0.307053,-0.302902,-0.297997,...,-0.031245,-0.132911,-0.121926,-0.124158,0.241199,-0.121665,-0.206861,-0.264959,-0.197120,-0.300088
NCHC,0.167334,0.143166,0.130094,0.056575,0.048420,0.057685,0.030785,0.186505,0.145850,0.124675,...,0.031999,0.042225,0.008273,0.098038,-0.015331,0.117978,0.054238,-0.111621,-0.049742,-0.086725


In [93]:
missing_player_cols = player_conference_baselines.columns.difference(conference_baseline)
missing_player_cols

Index(['1 Timers Pass from East-West', 'Base Successful OZ Blocked Passes',
       'Carry Controlled Entries with Play After',
       'Carry Controlled Entries with Scoring Chance After',
       'Carry Controlled Entries with Shot On Net After',
       'Carry-Out with Play After Rate', 'Carry-Outs with Play After',
       'Chip Ins Below Hash Marks', 'Controlled Entries',
       'Controlled Entries with Play After',
       ...
       'Total Offsides', 'Total Outlet Pass Attempts',
       'Total Pass to Slot Attempts', 'Total Reception Preventions',
       'Total Self Chip-In Recoveries', 'Total Shot From Slot Attempts',
       'Total South Pass Attempts in the NZ',
       'Total South Pass Attempts in the OZ',
       'Total Successful Shot From Slot Attempts', 'Total Time of Possession'],
      dtype='object', length=168)

In [98]:
for df in [player_conference_baselines, conference_baseline]:
    df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

print(player_conference_baselines.columns)

Index(['successful_oz_offensive_touches', 'failed_oz_offensive_touches',
       'failed_oz_possessions', 'successful_dz_offensive_touches',
       'failed_dz_offensive_touches', 'dz_offensive_touches_success_rate',
       'failed_dz_possessions', 'successful_nz_offensive_touches',
       'failed_nz_offensive_touches', 'failed_nz_possessions',
       ...
       'controlled_exit_with_play_after_rate',
       'successful_defensive_dump-in_recoveries',
       'failed_defensive_dump-in_recoveries', 'total_dump_out_attempts',
       'dump_out_rate', 'successful_dump_out_attempts', 'total_icings',
       'successful_oz_body_checks', 'successful_dz_body_checks',
       'successful_body_checks'],
      dtype='object', length=168)


In [106]:
common_cols = conference_baseline.columns.intersection(player_conference_baselines.columns)
len(common_cols)

168

# Subtracting Conference Baselines from Player Stat Lines

In [128]:
# last cleanup of conference_baseline df
junk_cols = [col for col in conference_baseline.columns if 'unnamed' in col]
conference_baseline.drop(columns=junk_cols, inplace=True, errors='ignore')

if 'conference' in conference_baseline.columns:
    conference_baseline.set_index('conference', inplace=True)

conference_baseline.index.name = 'conference'
conference_baseline

,successful_oz_open_ice_dekes,failed_oz_open_ice_dekes,successful_open_ice_dekes,failed_open_ice_dekes,dump_in_rate,total_dump_in_attempts,dump_ins_below_hash_marks,total_chip_in_attempts,chip_ins_below_hash_marks,total_self_chip-in_recoveries,...,successful_oz_blocked_passes,successful_nz_blocked_passes,successful_blocked_passes,dz_reception_preventions,total_reception_preventions,total_oz_obstruction_penalties,total_oz_penalties,total_dz_obstruction_penalties,total_nz_undisciplined_penalties,total_obstruction_penalties
conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,0.012793,0.044063,0.216652,0.286144,0.463172,0.561241,0.514828,0.475375,0.448341,0.140026,...,-0.020243,-0.021864,0.042433,0.145471,-0.131051,0.449798,0.409561,0.714577,-0.093930,0.837592
Big Ten,0.962282,0.733003,0.836184,0.619693,-1.301132,-0.487698,-0.471846,-0.752269,-0.755214,-0.926122,...,0.417228,0.407089,0.576650,0.487216,0.635648,0.259444,0.721831,-0.314219,0.643356,-0.008303
CCHA,-0.193569,-0.239019,-0.130594,-0.116506,0.707995,0.732614,0.682087,0.609535,0.593783,0.797617,...,0.588119,0.478901,0.670884,0.962295,0.916047,-0.823454,-0.599782,0.498416,0.167273,-0.048157
ECAC,-0.454442,-0.198017,-0.466706,-0.144880,0.001229,-0.489008,-0.505586,-0.399811,-0.408798,-0.481893,...,-0.607032,-0.662549,-0.759222,-0.679242,-0.553965,-0.290465,-0.611987,-0.027765,-0.598321,-0.309696
Hockey East,0.202873,-0.020888,0.140487,-0.171655,-0.204844,-0.211968,-0.122970,-0.048272,-0.017731,-0.076539,...,0.051067,-0.111981,-0.107158,-0.085905,-0.232441,0.183303,0.156920,-0.241082,-0.283077,-0.216176
Independent,-0.867945,-1.278178,-0.982803,-1.556967,0.498194,-0.717030,-0.737938,-0.460608,-0.398639,-0.306639,...,-1.253994,-0.632772,-1.373303,-0.826707,-0.857115,-0.669479,0.112982,-1.094349,1.176055,-1.191958
NCHC,0.271071,0.619598,0.236074,0.584541,-0.238689,0.332539,0.347250,0.295342,0.283651,0.673543,...,0.553487,0.600570,0.739676,-0.032943,0.234061,0.657071,0.144715,-0.108351,-0.072914,0.463299


In [126]:
player_deltas = player_scaled.copy()

player_deltas.columns = player_deltas.columns.str.lower().str.strip().str.replace(' ', '_')
player_deltas.set_index('conference', inplace=True)

In [127]:
player_deltas

,team,player,position,toi_(min),successful_oz_offensive_touches,failed_oz_offensive_touches,failed_oz_possessions,successful_dz_offensive_touches,failed_dz_offensive_touches,dz_offensive_touches_success_rate,...,controlled_exit_with_play_after_rate,successful_defensive_dump-in_recoveries,failed_defensive_dump-in_recoveries,total_dump_out_attempts,dump_out_rate,successful_dump_out_attempts,total_icings,successful_oz_body_checks,successful_dz_body_checks,successful_body_checks
conference,,,,,,,,,,,,,,,,,,,,,
Independent,Stonehill College Skyhawks,Zach Aben,F,135:14,-1.028896,-1.068593,-0.994559,-0.922229,-0.942070,-0.116737,...,-0.978175,-0.651678,-0.484882,-1.035419,0.155515,-1.018740,-0.725555,0.053540,-0.564963,-0.643342
Atlantic Hockey,College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,0.644482,0.449653,0.416347,-0.045624,-0.256116,0.774085,...,-0.090899,-0.482237,-0.164479,0.065895,0.088553,0.191495,0.062959,0.949538,-0.393749,-0.013428
Big Ten,Ohio State University Buckeyes,Chris Able,D,601:37,-0.571468,-0.636314,-0.790349,1.841215,1.206389,0.959765,...,0.268808,1.456926,1.277335,1.093789,1.576964,1.145014,0.536067,-0.842458,0.633537,0.144050
CCHA,Michigan Tech University Huskies,Ryan Abraham,F,77:05,-1.121540,-1.163483,-1.198769,-1.055857,-1.071495,-0.692682,...,-0.791380,-0.726985,-0.751885,-1.280155,-0.931178,-1.275457,-1.040960,-0.394459,-0.907392,-1.115777
Independent,University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,-0.415131,-0.056429,-0.140589,1.755692,1.542894,0.473231,...,0.086556,1.306311,1.277335,1.191683,0.942612,1.108340,0.851473,-0.842458,0.291109,-0.170907
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NCHC,University of North Dakota Fighting Hawks,Bennett Zmolek,D,513:08,-0.669902,-0.435990,-0.456187,1.108929,0.365125,1.355917,...,-0.655529,1.419272,0.743330,-0.325683,0.324556,-0.285264,0.062959,-0.842458,1.318394,1.088920
Atlantic Hockey,Army Black Knights,Easton Zueger,D,509:06,-0.710434,-0.172406,-0.103460,0.654591,1.866457,-1.587091,...,-0.927474,1.136870,2.185144,2.244050,1.586222,2.208554,2.270797,-0.842458,1.489609,0.931442
Hockey East,University of New Hampshire Wildcats,Conner de Haro,D,720:43,-0.009815,0.101722,0.156444,1.921392,1.982940,0.172745,...,0.523369,2.624189,2.452147,2.072735,1.027679,2.061859,1.797689,-0.394459,0.291109,0.301528


In [129]:

player_delta_means = player_deltas.groupby('conference').mean(numeric_only=True)
player_delta_means

,successful_oz_offensive_touches,failed_oz_offensive_touches,failed_oz_possessions,successful_dz_offensive_touches,failed_dz_offensive_touches,dz_offensive_touches_success_rate,failed_dz_possessions,successful_nz_offensive_touches,failed_nz_offensive_touches,failed_nz_possessions,...,controlled_exit_with_play_after_rate,successful_defensive_dump-in_recoveries,failed_defensive_dump-in_recoveries,total_dump_out_attempts,dump_out_rate,successful_dump_out_attempts,total_icings,successful_oz_body_checks,successful_dz_body_checks,successful_body_checks
conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,-0.066320,-0.062870,-0.057518,0.003647,0.019504,-0.011650,0.003376,-0.020664,-0.010561,-0.063493,...,0.001965,0.008726,0.034786,-0.002764,0.017649,-0.003433,-0.002086,0.039594,-0.002021,0.012307
Big Ten,0.211941,0.210393,0.230865,0.101884,0.052915,0.180144,0.076037,0.211108,0.187592,0.251013,...,0.108128,0.038474,-0.023483,-0.009458,-0.227258,-0.013106,0.062959,0.088908,0.252060,0.308435
CCHA,-0.039919,-0.044952,-0.022055,0.013670,0.017452,0.051482,0.019229,-0.010674,-0.005530,0.002954,...,0.085231,0.015425,0.016043,0.053984,0.034468,0.065895,-0.011706,0.132832,0.075954,0.109210
ECAC,-0.025984,-0.042837,-0.061911,-0.043933,-0.001844,-0.160878,-0.017501,-0.052202,-0.006963,0.005458,...,-0.260248,-0.026807,-0.005186,-0.075536,-0.052396,-0.094037,0.059740,-0.040936,-0.072285,-0.106094
Hockey East,-0.003437,0.034617,0.035425,0.041426,-0.006365,0.085349,0.004969,-0.028744,-0.061695,-0.061469,...,0.127090,0.019081,0.022423,0.038846,0.037571,0.040388,-0.015893,0.088908,0.002748,0.055838
Independent,-0.260016,-0.280146,-0.290701,-0.221805,-0.188269,-0.134684,-0.150866,-0.307053,-0.302902,-0.297997,...,-0.031245,-0.132911,-0.121926,-0.124158,0.241199,-0.121665,-0.206861,-0.264959,-0.197120,-0.300088
NCHC,0.167334,0.143166,0.130094,0.056575,0.048420,0.057685,0.030785,0.186505,0.145850,0.124675,...,0.031999,0.042225,0.008273,0.098038,-0.015331,0.117978,0.054238,-0.111621,-0.049742,-0.086725


In [132]:
common_cols = conference_baseline.columns.intersection(player_deltas.columns)

player_deltas.reset_index(inplace=True)

aligned_baselines = player_deltas[['conference']].merge(conference_baseline, on='conference', how='left')

player_deltas[common_cols] = player_deltas[common_cols] - aligned_baselines[common_cols]

player_deltas

/var/folders/g1/5jyzgjs54vj6npmtcwwyq7cr0000gn/T/ipykernel_87121/1921990081.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  player_deltas.reset_index(inplace=True)


,index,conference,team,player,position,toi_(min),successful_oz_offensive_touches,failed_oz_offensive_touches,failed_oz_possessions,successful_dz_offensive_touches,...,controlled_exit_with_play_after_rate,successful_defensive_dump-in_recoveries,failed_defensive_dump-in_recoveries,total_dump_out_attempts,dump_out_rate,successful_dump_out_attempts,total_icings,successful_oz_body_checks,successful_dz_body_checks,successful_body_checks
0,0,Independent,Stonehill College Skyhawks,Zach Aben,F,135:14,1.325179,1.546329,1.949394,2.394185,...,-1.457838,1.736715,1.178641,-0.131162,-1.731198,-0.164045,1.770301,2.101965,1.340276,1.722580
1,1,Atlantic Hockey,College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,0.727986,0.390310,0.267962,-1.030919,...,0.267067,-1.250064,-1.622353,-0.785673,-0.078183,-0.610080,-0.638005,0.268042,-0.843774,-0.664871
2,2,Big Ten,Ohio State University Buckeyes,Chris Able,D,601:37,-1.558633,-1.705093,-2.138877,1.876218,...,-0.756475,2.040267,3.020268,2.716646,3.807273,2.867365,1.213134,-0.655342,-1.305897,-1.478943
3,3,CCHA,Michigan Tech University Huskies,Ryan Abraham,F,77:05,-0.997509,-1.024861,-1.347051,-1.963637,...,-1.116167,-1.877239,-2.304778,-2.407850,-1.438295,-2.485082,-1.219447,-1.706732,-2.035326,-2.401076
4,4,Independent,University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,1.938944,2.558493,2.803363,5.072106,...,-0.393106,3.694705,2.940858,2.095940,-0.944100,1.963035,3.347329,1.205968,2.196348,2.195015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1543,1543,NCHC,University of North Dakota Fighting Hawks,Bennett Zmolek,D,513:08,-2.194194,-1.719715,-1.675787,0.307819,...,-0.569446,0.357819,0.273661,-1.271606,0.105487,-1.405361,-0.445948,0.223610,2.075029,2.088909
1544,1544,Atlantic Hockey,Army Black Knights,Easton Zueger,D,509:06,-0.626929,-0.231749,-0.251846,-0.330704,...,-0.569508,0.369043,0.727270,1.392482,1.419485,1.406979,1.569833,-1.523953,1.039583,0.280000
1545,1545,Hockey East,University of New Hampshire Wildcats,Conner de Haro,D,720:43,0.260333,-0.008740,0.032700,1.688992,...,-0.901500,2.752881,2.692966,1.938664,0.606657,1.893655,2.282860,-1.025191,0.432274,-0.013461
1546,1546,Atlantic Hockey,Robert Morris University Colonials,Luke van Why,D,593:08,0.189495,0.052922,-0.177588,0.070182,...,0.905906,0.726752,0.246666,0.389062,0.649373,0.343439,0.465914,-1.523953,1.210798,0.752435


In [133]:
player_deltas.to_csv('ncaa_d1_player_deltas.csv')